# LeetCode #18 — 4Sum: từ 3Sum tới k-Sum tổng quát

**Mức độ:** Medium · **Chủ đề:** Array, Two Pointers, Sorting, Recursion

Skeleton trên LeetCode:

```python
class Solution:
    def fourSum(self, nums: List[int], target: int) -> List[List[int]]:
```

Bài này là bước tiếp theo sau #15 3Sum: thêm **một vòng lặp cố định** nữa. Nhưng mục tiêu thật sự của notebook là giúp bạn thấy **quy luật chung** và viết được hàm **k-Sum đệ quy** — giải 2Sum, 3Sum, 4Sum, … bằng cùng một đoạn code.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách mở rộng khung 3Sum lên 4Sum: hai vòng cố định + Two Pointers.
2. Bỏ trùng ở **mỗi tầng** cố định.
3. Hai mẹo **cắt tỉa** dựa trên giá trị nhỏ nhất / lớn nhất có thể.
4. Hàm **k-Sum đệ quy**: đưa bài k số về bài (k-1) số.
5. Vì sao bài này phải để ý **tràn số** trong ngôn ngữ khác Python.

# 1. Bài toán là gì?

Cho mảng `nums` gồm `n` số nguyên và `target`. Trả về mọi bộ bốn **không trùng nhau** `[nums[a], nums[b], nums[c], nums[d]]` sao cho:

- `a, b, c, d` đôi một khác nhau,
- `nums[a] + nums[b] + nums[c] + nums[d] == target`.

Thứ tự kết quả tùy ý.

**Constraints (theo đề):**

- `1 <= nums.length <= 200`
- `-10^9 <= nums[i] <= 10^9`
- `-10^9 <= target <= 10^9`

---

## Ví dụ

```text
nums = [1, 0, -1, 0, -2, 2], target = 0
-> [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]

nums = [2, 2, 2, 2, 2], target = 8
-> [[2, 2, 2, 2]]
```

Chú ý: `n` có thể nhỏ hơn 4 — khi đó trả về `[]`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Sắp xếp (sorting)
2. Hai con trỏ (two pointers)
3. Đệ quy (recursion)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [1]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [2]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [3]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [4]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [5]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [6]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [7]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [8]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [9]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [10]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.4 Áp dụng vào bài này

Mở rộng 3Sum: **sắp xếp**, cố định hai số (hoặc viết hàm **đệ quy** k-Sum giảm `k` mỗi tầng), tới `k = 2` thì dùng **hai con trỏ**. Bỏ trùng ở mỗi tầng giống như bỏ trùng trong quay lui.

# 3. Nhắc lại khung 3Sum

```text
sort
for i:                         <- cố định số thứ nhất (bỏ trùng)
    left, right = i+1, n-1     <- Two Pointers tìm 2 số còn lại
    ...
```

4Sum chỉ cần **thêm một tầng**:

```text
sort
for i:                         <- số thứ nhất (bỏ trùng)
    for j > i:                 <- số thứ hai (bỏ trùng)
        left, right = j+1, n-1 <- Two Pointers
```

# 4. Cách dễ nghĩ nhất: thử mọi bộ bốn

In [11]:
from typing import List
from itertools import combinations

def four_sum_bruteforce(nums: List[int], target: int) -> List[List[int]]:
    found = set()
    for quad in combinations(nums, 4):
        if sum(quad) == target:
            found.add(tuple(sorted(quad)))
    return [list(t) for t in sorted(found)]

print(four_sum_bruteforce([1, 0, -1, 0, -2, 2], 0))
print(four_sum_bruteforce([2, 2, 2, 2, 2], 8))

[[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]
[[2, 2, 2, 2]]


`combinations(nums, 4)` sinh mọi cách chọn 4 phần tử (theo vị trí). Số bộ là `C(n, 4) ≈ n⁴ / 24`. Với `n = 200` khoảng `6.5 * 10^7` — quá chậm trong Python, và độ phức tạp `O(n⁴)`.

# 5. Lời giải: hai vòng cố định + Two Pointers

Bỏ trùng ở **ba chỗ**, giống 3Sum:

1. `i`: bỏ qua nếu `nums[i] == nums[i - 1]`.
2. `j`: bỏ qua nếu `j > i + 1` **và** `nums[j] == nums[j - 1]`.
   (Điều kiện `j > i + 1` quan trọng: phần tử đầu tiên của tầng `j` luôn được xét, kể cả khi nó bằng `nums[i]` — ví dụ bộ `[2, 2, 2, 2]`.)
3. Sau khi ghi nhận: dời `left` / `right` qua các giá trị trùng.

In [12]:
def four_sum(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []

    for i in range(n - 3):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        for j in range(i + 1, n - 2):
            if j > i + 1 and nums[j] == nums[j - 1]:
                continue
            left, right = j + 1, n - 1
            while left < right:
                total = nums[i] + nums[j] + nums[left] + nums[right]
                if total < target:
                    left += 1
                elif total > target:
                    right -= 1
                else:
                    result.append([nums[i], nums[j], nums[left], nums[right]])
                    left += 1
                    right -= 1
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1
                    while left < right and nums[right] == nums[right + 1]:
                        right -= 1
    return result

print(four_sum([1, 0, -1, 0, -2, 2], 0))
print(four_sum([2, 2, 2, 2, 2], 8))
print(four_sum([1, 2, 3], 6))

[[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]
[[2, 2, 2, 2]]
[]


# 6. Mô phỏng: chỉ in các lần ghi nhận và bỏ qua

`nums = [1, 0, -1, 0, -2, 2]` → sắp xếp `[-2, -1, 0, 0, 1, 2]`, `target = 0`.

In [13]:
def trace_four_sum(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    print("Sắp xếp:", nums)
    n = len(nums)
    result = []
    for i in range(n - 3):
        if i > 0 and nums[i] == nums[i - 1]:
            print(f"i={i} ({nums[i]}): trùng -> bỏ qua")
            continue
        for j in range(i + 1, n - 2):
            if j > i + 1 and nums[j] == nums[j - 1]:
                print(f"  i={i} j={j} ({nums[j]}): trùng -> bỏ qua")
                continue
            left, right = j + 1, n - 1
            steps = 0
            while left < right:
                steps += 1
                total = nums[i] + nums[j] + nums[left] + nums[right]
                if total < target:
                    left += 1
                elif total > target:
                    right -= 1
                else:
                    quad = [nums[i], nums[j], nums[left], nums[right]]
                    result.append(quad)
                    print(f"  i={i} j={j} left={left} right={right}: GHI {quad}")
                    left += 1
                    right -= 1
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1
                    while left < right and nums[right] == nums[right + 1]:
                        right -= 1
            print(f"  i={i}({nums[i]}) j={j}({nums[j]}): hai con trỏ chạy {steps} bước")
    return result

print("Kết quả:", trace_four_sum([1, 0, -1, 0, -2, 2], 0))

Sắp xếp: [-2, -1, 0, 0, 1, 2]
  i=0 j=1 left=4 right=5: GHI [-2, -1, 1, 2]
  i=0(-2) j=1(-1): hai con trỏ chạy 3 bước
  i=0 j=2 left=3 right=5: GHI [-2, 0, 0, 2]
  i=0(-2) j=2(0): hai con trỏ chạy 1 bước
  i=0 j=3 (0): trùng -> bỏ qua
  i=1 j=2 left=3 right=4: GHI [-1, 0, 0, 1]
  i=1(-1) j=2(0): hai con trỏ chạy 2 bước
  i=1 j=3 (0): trùng -> bỏ qua
  i=2(0) j=3(0): hai con trỏ chạy 1 bước
Kết quả: [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]


Các dòng "trùng → bỏ qua" là bỏ trùng ở tầng `j`: giá trị `0` ở `j = 3` đã được dùng làm số thứ hai ở `j = 2`, nên mọi bộ bốn bắt đầu bằng `(nums[i], 0)` đều đã được tìm.

Trong ví dụ trên, bỏ qua chỉ giúp tiết kiệm thời gian. Nhưng với input như `[-1, 0, 0, 0, 1]`, thiếu nó sẽ sinh **bộ trùng**:

In [14]:
def four_sum_no_j_skip(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 3):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        for j in range(i + 1, n - 2):             # không bỏ trùng tầng j
            left, right = j + 1, n - 1
            while left < right:
                total = nums[i] + nums[j] + nums[left] + nums[right]
                if total < target:
                    left += 1
                elif total > target:
                    right -= 1
                else:
                    result.append([nums[i], nums[j], nums[left], nums[right]])
                    left += 1
                    right -= 1
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1
                    while left < right and nums[right] == nums[right + 1]:
                        right -= 1
    return result

print("Thiếu bỏ trùng tầng j:", four_sum_no_j_skip([-1, 0, 0, 0, 1], 0))
print("Đúng                 :", four_sum([-1, 0, 0, 0, 1], 0))

Thiếu bỏ trùng tầng j: [[-1, 0, 0, 1], [-1, 0, 0, 1]]
Đúng                 : [[-1, 0, 0, 1]]


# 7. Cắt tỉa (pruning) cho nhanh hơn

Vì mảng đã sắp xếp, với `i` cố định:

- **Tổng nhỏ nhất** dùng `nums[i]` là `nums[i] + nums[i+1] + nums[i+2] + nums[i+3]`. Nếu nó **> target**, mọi `i` lớn hơn còn tệ hơn → `break`.
- **Tổng lớn nhất** dùng `nums[i]` là `nums[i] + nums[n-3] + nums[n-2] + nums[n-1]`. Nếu nó **< target**, `nums[i]` quá nhỏ → `continue` sang `i` kế.

Tương tự cho tầng `j`. Các mẹo này không đổi độ phức tạp `O(n³)` nhưng giúp chạy nhanh hơn rõ rệt trên nhiều input.

In [15]:
def four_sum_pruned(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 3):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        if nums[i] + nums[i + 1] + nums[i + 2] + nums[i + 3] > target:
            break
        if nums[i] + nums[n - 3] + nums[n - 2] + nums[n - 1] < target:
            continue
        for j in range(i + 1, n - 2):
            if j > i + 1 and nums[j] == nums[j - 1]:
                continue
            if nums[i] + nums[j] + nums[j + 1] + nums[j + 2] > target:
                break
            if nums[i] + nums[j] + nums[n - 2] + nums[n - 1] < target:
                continue
            left, right = j + 1, n - 1
            while left < right:
                total = nums[i] + nums[j] + nums[left] + nums[right]
                if total < target:
                    left += 1
                elif total > target:
                    right -= 1
                else:
                    result.append([nums[i], nums[j], nums[left], nums[right]])
                    left += 1
                    right -= 1
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1
                    while left < right and nums[right] == nums[right + 1]:
                        right -= 1
    return result

print(four_sum_pruned([1, 0, -1, 0, -2, 2], 0))

[[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]

# 8. Tổng quát hóa: k-Sum đệ quy

Quan sát:

- **2Sum** (mảng đã sắp xếp) → Two Pointers.
- **kSum** với `k > 2` → cố định một số `nums[i]`, giải **(k-1)Sum** trên phần còn lại với `target - nums[i]`.

```python
def k_sum(nums, target, k, start):
    if k == 2:
        return two_sum(nums, target, start)       # base case
    res = []
    for i in range(start, len(nums) - k + 1):
        if i > start and nums[i] == nums[i-1]:    # bỏ trùng ở tầng này
            continue
        for rest in k_sum(nums, target - nums[i], k - 1, i + 1):
            res.append([nums[i]] + rest)
    return res
```

Điều kiện bỏ trùng `i > start` là phiên bản tổng quát của `i > 0` (tầng 1) và `j > i + 1` (tầng 2).

In [16]:
def two_sum_sorted_all(nums: List[int], target: int, start: int) -> List[List[int]]:
    res = []
    left, right = start, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total < target:
            left += 1
        elif total > target:
            right -= 1
        else:
            res.append([nums[left], nums[right]])
            left += 1
            right -= 1
            while left < right and nums[left] == nums[left - 1]:
                left += 1
            while left < right and nums[right] == nums[right + 1]:
                right -= 1
    return res


def k_sum(nums: List[int], target: int, k: int, start: int = 0) -> List[List[int]]:
    if k == 2:
        return two_sum_sorted_all(nums, target, start)
    res = []
    for i in range(start, len(nums) - k + 1):
        if i > start and nums[i] == nums[i - 1]:
            continue
        for rest in k_sum(nums, target - nums[i], k - 1, i + 1):
            res.append([nums[i]] + rest)
    return res


nums = sorted([1, 0, -1, 0, -2, 2])
print("2Sum:", k_sum(nums, 0, 2))
print("3Sum:", k_sum(nums, 0, 3))
print("4Sum:", k_sum(nums, 0, 4))
print("5Sum:", k_sum(nums, 0, 5))

2Sum:

 [[-2, 2], [-1, 1], [0, 0]]
3Sum: [[-2, 0, 2], [-1, 0, 1]]
4Sum: [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]
5Sum: [[-2, -1, 0, 1, 2]]


# 9. Phiên bản đúng format LeetCode

Ta nộp bản hai vòng lặp (dễ đọc, chạy nhanh). Bản k-Sum cũng được chấp nhận.

In [17]:
class Solution:
    def fourSum(self, nums: List[int], target: int) -> List[List[int]]:
        nums.sort()
        n = len(nums)
        result = []
        for i in range(n - 3):
            if i > 0 and nums[i] == nums[i - 1]:
                continue
            for j in range(i + 1, n - 2):
                if j > i + 1 and nums[j] == nums[j - 1]:
                    continue
                left, right = j + 1, n - 1
                while left < right:
                    total = nums[i] + nums[j] + nums[left] + nums[right]
                    if total < target:
                        left += 1
                    elif total > target:
                        right -= 1
                    else:
                        result.append([nums[i], nums[j], nums[left], nums[right]])
                        left += 1
                        right -= 1
                        while left < right and nums[left] == nums[left - 1]:
                            left += 1
                        while left < right and nums[right] == nums[right + 1]:
                            right -= 1
        return result

In [18]:
def normalize(quads):
    return sorted(sorted(q) for q in quads)

def check(name, actual, expected):
    assert normalize(actual) == normalize(expected), (name, actual, expected)
    assert len(actual) == len({tuple(sorted(q)) for q in actual}), (name, "có bộ trùng")
    print(f"{name:<28} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.fourSum([1, 0, -1, 0, -2, 2], 0), [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]])
check("Ví dụ 2", sol.fourSum([2, 2, 2, 2, 2], 8), [[2, 2, 2, 2]])
check("Ít hơn 4 phần tử", sol.fourSum([1, 2, 3], 6), [])
check("Không có đáp án", sol.fourSum([1, 2, 3, 4], 100), [])
check("Số rất lớn", sol.fourSum([10**9] * 4, -294967296), [])
check("Số rất lớn, có đáp án", sol.fourSum([10**9] * 4, 4 * 10**9), [[10**9] * 4])

Ví dụ 1                      PASS  [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]


Ví dụ 2                      PASS  [[2, 2, 2, 2]]
Ít hơn 4 phần tử             PASS  []
Không có đáp án              PASS  []
Số rất lớn                   PASS  []
Số rất lớn, có đáp án        PASS  [[1000000000, 1000000000, 1000000000, 1000000000]]


Test "Số rất lớn": tổng thật là `4 * 10^9`. Trong C++/Java dùng `int` 32-bit, tổng này **tràn** và quay vòng thành `-294967296` — một lời giải sai sẽ "tìm thấy" bộ bốn này. Python không tràn nên trả về `[]` đúng. Khi viết bằng ngôn ngữ khác, hãy tính tổng bằng kiểu 64-bit (`long`).

## Đối chiếu nhiều cách

In [19]:
import random

random.seed(18)
for _ in range(1500):
    nums = [random.randint(-5, 5) for _ in range(random.randint(1, 11))]
    target = random.randint(-8, 8)
    expected = normalize(four_sum_bruteforce(nums, target))
    assert normalize(sol.fourSum(nums[:], target)) == expected, (nums, target)
    assert normalize(four_sum_pruned(nums, target)) == expected, (nums, target)
    assert normalize(k_sum(sorted(nums), target, 4)) == expected, (nums, target)
print("1500 test ngẫu nhiên: 4 cách khớp nhau")

1500 test ngẫu nhiên: 4 cách khớp nhau


# 10. Vì sao đúng? Độ phức tạp?

**Đúng:** với mỗi cặp giá trị `(nums[i], nums[j])` (mỗi giá trị chỉ làm "số cố định" một lần ở mỗi tầng), Two Pointers tìm **mọi** cặp `(c, d)` còn lại có tổng phù hợp, không trùng — lập luận y hệt 3Sum. Bỏ trùng ở từng tầng đảm bảo cùng một bộ giá trị chỉ được sinh một lần.

**Độ phức tạp** (`n = len(nums)`):

```text
Time:  O(n log n) + O(n²) cặp (i, j) × O(n) hai con trỏ = O(n³)
Space: O(1) bộ nhớ phụ (không tính kết quả và sort)
```

**k-Sum tổng quát:** `O(n^(k-1))` thời gian, `O(k)` độ sâu đệ quy.

| Bài | Số tầng cố định | Thời gian |
|---|---|---|
| 2Sum (đã sắp xếp) | 0 | O(n) |
| 3Sum | 1 | O(n²) |
| 4Sum | 2 | O(n³) |
| kSum | k − 2 | O(n^(k−1)) |

# 11. Những lỗi dễ mắc

## Lỗi 1: Bỏ trùng tầng `j` bằng `j > 0` thay vì `j > i + 1`

Với `[2, 2, 2, 2, 2]`: ở `i = 0`, `j = 1` có `nums[1] == nums[0]` → bị bỏ qua sai, mất bộ `[2, 2, 2, 2]`.

In [20]:
def wrong_skip_j(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 3):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        for j in range(i + 1, n - 2):
            if j > 0 and nums[j] == nums[j - 1]:     # SAI: phải là j > i + 1
                continue
            left, right = j + 1, n - 1
            while left < right:
                total = nums[i] + nums[j] + nums[left] + nums[right]
                if total < target:
                    left += 1
                elif total > target:
                    right -= 1
                else:
                    result.append([nums[i], nums[j], nums[left], nums[right]])
                    left += 1
                    right -= 1
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1
    return result

print("Sai :", wrong_skip_j([2, 2, 2, 2, 2], 8))
print("Đúng:", four_sum([2, 2, 2, 2, 2], 8))

Sai : []
Đúng: [[2, 2, 2, 2]]


## Lỗi 2: Cắt tỉa bằng `nums[i] > target` như 3Sum

Ở 3Sum, `target = 0` nên "số nhỏ nhất > 0 thì dừng" là đúng. Ở 4Sum, `target` tùy ý và có thể **âm**: ví dụ `nums = [-5, -4, -3, -2]`, `target = -14`, ta có `nums[0] = -5 > -14` nhưng `-5 + -4 + -3 + -2 = -14` là đáp án!

In [21]:
def wrong_prune(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    if nums and nums[0] > target:        # SAI: chỉ đúng khi mọi số không âm
        return []
    return four_sum(nums, target)

print("Sai :", wrong_prune([-5, -4, -3, -2], -14))
print("Đúng:", four_sum([-5, -4, -3, -2], -14))

Sai : []
Đúng: [[-5, -4, -3, -2]]


Cắt tỉa đúng phải dựa trên **tổng nhỏ nhất có thể** (mục 7), không phải một phần tử.

## Lỗi 3: Tràn số (ngôn ngữ khác Python)

`nums[i]` tới `10^9`, tổng 4 số tới `4 * 10^9` > `2^31 - 1`. Dùng `long` trong Java/C++.

## Lỗi 4: Vòng lặp chạy quá biên

`for i in range(n - 3)` và `for j in range(i + 1, n - 2)` để còn đủ chỗ cho 4 phần tử. Nếu `n < 4`, `range` rỗng → trả `[]` tự nhiên.

# 12. Mẹo nhận diện pattern

- "Tìm mọi bộ **k phần tử** có tổng bằng target, không trùng" → **sắp xếp + (k-2) tầng cố định + Two Pointers**, hoặc hàm **k-Sum đệ quy**.
- Bỏ trùng ở mỗi tầng: `i > start and nums[i] == nums[i - 1]`.
- Cắt tỉa theo **tổng nhỏ nhất / lớn nhất có thể** từ vị trí hiện tại.

Bài liên quan: **#1**, **#15**, **#16**, **#454 4Sum II** (bốn mảng riêng — dùng Hash Map đếm tổng từng cặp, `O(n²)`).

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [-3, -1, 0, 2, 4, 5]`, `target = 2`, tìm mọi bộ bốn. Gợi ý: liệt kê theo `nums[i]` tăng dần.

## Bài 2 — Đoán output

```python
print(list(range(5 - 3)))
print(len(list(combinations(range(6), 4))))
print(k_sum([1, 1, 1, 1, 1], 3, 3))
```

## Bài 3 — Điền chỗ trống

In [22]:
def exercise_3_four_sum(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 3):
        # TODO 1: bỏ trùng tầng i
        for j in range(i + 1, n - 2):
            # TODO 2: bỏ trùng tầng j (cẩn thận điều kiện!)
            left, right = j + 1, n - 1
            # TODO 3: Two Pointers + bỏ trùng sau khi ghi nhận
    return result


# assert normalize(exercise_3_four_sum([1, 0, -1, 0, -2, 2], 0)) == [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]
# assert exercise_3_four_sum([2, 2, 2, 2, 2], 8) == [[2, 2, 2, 2]]

## Bài 4 — Biến thể: 4Sum II (LeetCode #454)

Cho bốn mảng `A, B, C, D` cùng độ dài `n`. Đếm số bộ chỉ số `(i, j, k, l)` sao cho `A[i] + B[j] + C[k] + D[l] == 0`. Yêu cầu `O(n²)`.

```python
four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
four_sum_count([0], [0], [0], [0])                -> 1
```

In [23]:
def exercise_4_four_sum_count(A, B, C, D) -> int:
    # TODO
    pass


# assert exercise_4_four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2
# assert exercise_4_four_sum_count([0], [0], [0], [0]) == 1

## Bài 5 — Mini challenge: k-Sum có cắt tỉa

Viết `k_sum_pruned(nums, target, k)` (nums đã sắp xếp) giống `k_sum` nhưng thêm cắt tỉa ở mỗi tầng:

- Nếu `nums[i] * k > target` → `break` (k số nhỏ nhất từ `i` đều ≥ `nums[i]`).
- Nếu `nums[-1] * k < target` → không thể đạt, trả `[]` ngay.

(Cả hai đúng vì mảng đã sắp xếp.) Kiểm tra kết quả khớp `k_sum`.

In [24]:
def exercise_5_k_sum_pruned(nums: List[int], target: int, k: int, start: int = 0) -> List[List[int]]:
    # TODO
    pass


# assert normalize(exercise_5_k_sum_pruned(sorted([1, 0, -1, 0, -2, 2]), 0, 4)) == normalize(k_sum(sorted([1, 0, -1, 0, -2, 2]), 0, 4))
# assert exercise_5_k_sum_pruned([2, 2, 2, 2, 2], 8, 4) == [[2, 2, 2, 2]]

# 14. Gợi ý

**Bài 1**
- Mức 1: `nums[i] = -3`: cần 3 số tổng 5 trong `[-1, 0, 2, 4, 5]`.
- Mức 2: tìm bộ ba tổng 5 trong `[-1, 0, 2, 4, 5]` bằng Two Pointers như 3Sum.
- Mức 3: `nums[i] = -1`: cần 3 số tổng 3 trong `[0, 2, 4, 5]` — không có.

**Bài 3**
- Mức 1: so với mục 5.
- Mức 2: tầng `j` dùng `j > i + 1`.

**Bài 4**
- Mức 1: tách thành hai nửa: `A + B` và `C + D`.
- Mức 2: đếm tần suất mọi tổng `a + b` vào `Counter`.
- Mức 3: với mỗi `c + d`, cộng `count[-(c + d)]` vào đáp án.

**Bài 5**
- Mức 1: kiểm tra `nums[-1] * k < target` ở đầu hàm (sau base case hoặc trước).
- Mức 2: trong vòng `for i`, kiểm tra `nums[i] * k > target` → `break`.
- Mức 3: cẩn thận khi `start >= len(nums)`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
print(four_sum([-3, -1, 0, 2, 4, 5], 2))

[[-3, -1, 2, 4]]


## Đáp án Bài 2

In [26]:
print(list(range(5 - 3)))                       # [0, 1]
print(len(list(combinations(range(6), 4))))     # C(6, 4) = 15
print(k_sum([1, 1, 1, 1, 1], 3, 3))             # [[1, 1, 1]]

[0, 1]
15
[[1, 1, 1]]


## Đáp án Bài 3

In [27]:
def answer_3_four_sum(nums: List[int], target: int) -> List[List[int]]:
    return four_sum(nums, target)

assert normalize(answer_3_four_sum([1, 0, -1, 0, -2, 2], 0)) == [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]
assert answer_3_four_sum([2, 2, 2, 2, 2], 8) == [[2, 2, 2, 2]]
print("Bài 3 PASS (đáp án giống hàm four_sum ở mục 5)")

Bài 3 PASS (đáp án giống hàm four_sum ở mục 5)


## Đáp án Bài 4

Đây là kỹ thuật **gặp nhau ở giữa (meet in the middle)**: thay vì `n⁴` bộ, ta ghép `n²` tổng của nửa đầu với `n²` tổng của nửa sau qua Hash Map.

In [28]:
from collections import Counter

def answer_4_four_sum_count(A, B, C, D) -> int:
    ab = Counter(a + b for a in A for b in B)
    return sum(ab[-(c + d)] for c in C for d in D)

assert answer_4_four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2
assert answer_4_four_sum_count([0], [0], [0], [0]) == 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [29]:
def answer_5_k_sum_pruned(nums: List[int], target: int, k: int, start: int = 0) -> List[List[int]]:
    if start >= len(nums) or nums[-1] * k < target:
        return []
    if k == 2:
        return two_sum_sorted_all(nums, target, start)
    res = []
    for i in range(start, len(nums) - k + 1):
        if nums[i] * k > target:
            break
        if i > start and nums[i] == nums[i - 1]:
            continue
        for rest in answer_5_k_sum_pruned(nums, target - nums[i], k - 1, i + 1):
            res.append([nums[i]] + rest)
    return res

assert normalize(answer_5_k_sum_pruned(sorted([1, 0, -1, 0, -2, 2]), 0, 4)) == normalize(k_sum(sorted([1, 0, -1, 0, -2, 2]), 0, 4))
assert answer_5_k_sum_pruned([2, 2, 2, 2, 2], 8, 4) == [[2, 2, 2, 2]]
for _ in range(1000):
    nums = sorted(random.randint(-6, 6) for _ in range(random.randint(1, 10)))
    t, k = random.randint(-8, 8), random.randint(2, 5)
    assert normalize(answer_5_k_sum_pruned(nums, t, k)) == normalize(k_sum(nums, t, k)), (nums, t, k)
print("Bài 5 PASS (kể cả 1000 test ngẫu nhiên với k = 2..5)")

Bài 5 PASS (kể cả 1000 test ngẫu nhiên với k = 2..5)


# 16. Tổng kết

> **k-Sum = (k − 2) tầng cố định + Two Pointers.** Mỗi tầng bỏ trùng bằng `i > start and nums[i] == nums[i - 1]`. Cắt tỉa theo tổng nhỏ nhất/lớn nhất có thể.

---

## Checklist trước khi nộp

- [ ] Đã sắp xếp?
- [ ] Bỏ trùng tầng `j` bằng `j > i + 1` (không phải `j > 0`)?
- [ ] Không dùng cắt tỉa kiểu 3Sum (`nums[i] > 0`) khi target tùy ý?
- [ ] Ngôn ngữ khác Python: tổng dùng kiểu 64-bit?
- [ ] Đã thử `n < 4`, `[2,2,2,2,2]`, target âm?